# 01 - Download, Extração, Conversão em Parquet e Divisão Temporal do IoT-23
Este notebook baixa os dados oficiais do IoT-23 (versão leve: conn.log.labeled), realiza a leitura em streaming por captura, armazena em Parquet e define a divisão temporal provisória para o Cenário T.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import DATA_DIR, PARQUET_DIR, set_seed, SEED
from src.data import (
    download_iot23_lightweight,
    list_iot23_captures,
    convert_capture_to_parquet,
    read_conn_log_batches,
    generate_temporal_splits
)

set_seed(SEED)

# Parâmetros configuráveis da partição temporal provisória
TRAIN_RATIO = 0.70  # 70% capturas mais antigas para treino, 30% mais recentes para teste
MAX_EXPLORATION_ROWS_PER_CAPTURE = 50_000  # Limite opcional para exploração rápida se desejado (None = tudo)

In [ ]:
# Baixa e extrai caso não existam
download_iot23_lightweight(dest_dir=DATA_DIR)

In [ ]:
# Varre os logs e deriva as datas cronológicas do tráfego a partir de min(ts)
df_captures = list_iot23_captures(base_dir=DATA_DIR)
print(f"Total de capturas identificadas: {len(df_captures)}")
display(df_captures[["capture_name", "size_mb", "total_lines_approx", "derived_start_date"]])

In [ ]:
# Converte cada captura individualmente para Parquet (baixo uso de RAM)
for idx, row in df_captures.iterrows():
    raw_path = Path(row["filepath"])
    out_parquet = PARQUET_DIR / f"{row['capture_name']}.parquet"
    if not out_parquet.exists():
        print(f"Convertendo [{idx+1}/{len(df_captures)}]: {row['capture_name']}...")
        convert_capture_to_parquet(
            capture_log_path=raw_path,
            output_parquet_path=out_parquet,
            chunksize=200_000,
            max_rows=None
        )
print("✓ Todas as capturas foram salvas em formato Parquet.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

stats = []
for p_file in sorted(PARQUET_DIR.glob("*.parquet")):
    df_chunk = pd.read_parquet(p_file, columns=["label", "detailed_label", "id.orig_h", "id.resp_h"])
    tot = len(df_chunk)
    n_mal = (df_chunk["label"] == "Malicious").sum()
    stats.append({
        "capture": p_file.stem,
        "total_flows": tot,
        "malicious_pct": (n_mal / tot * 100) if tot > 0 else 0,
        "unique_src_ips": df_chunk["id.orig_h"].nunique(),
        "unique_dst_ips": df_chunk["id.resp_h"].nunique(),
        "attacks": list(df_chunk[df_chunk["label"] == "Malicious"]["detailed_label"].unique())
    })

df_stats = pd.DataFrame(stats)
display(df_stats.drop(columns=["attacks"]))

# Gráfico de proporção de tráfego malicioso por captura
plt.figure(figsize=(12, 4))
plt.bar(df_stats["capture"], df_stats["malicious_pct"], color="crimson", alpha=0.7)
plt.xticks(rotation=90)
plt.ylabel("% Malicioso")
plt.title("Proporção de Tráfego Malicioso por Captura do IoT-23")
plt.tight_layout()
plt.show()

In [ ]:
train_caps, test_caps = generate_temporal_splits(df_captures, train_ratio=TRAIN_RATIO)

splits_content = f"""# Divisão Temporal Provisória (Cenário T - IoT-23)

- **Critério de Divisão:** Ordenação cronológica estrita baseada no primeiro timestamp real (`min(ts)`) extraído do `conn.log.labeled`.
- **Proporção Alvo:** {int(TRAIN_RATIO*100)}% capturas mais antigas para treino, {int((1-TRAIN_RATIO)*100)}% mais recentes para teste.

## Capturas de Treino ({len(train_caps)})
{chr(10).join([f'- `{c}`' for c in train_caps])}

## Capturas de Teste ({len(test_caps)})
{chr(10).join([f'- `{c}`' for c in test_caps])}

## Justificativa Metodológica
Esta divisão previne vazamento de dados temporal (*data leakage*), simulando o cenário realista onde o modelo é treinado em redes infectadas conhecidas no passado e precisa detectar comportamentos e novos padrões de ataque em períodos posteriores.
"""

splits_path = repo_root / "splits.md"
splits_path.write_text(splits_content, encoding="utf-8")
print(f"✓ Arquivo {splits_path} gerado com sucesso.")

In [ ]:
# Commit das alterações de código e documentação (nunca dados)
!git status
!git add src/ tests/ splits.md notebooks/*.ipynb README.md requirements.txt .gitignore
!git commit -m "feat: configuracao dia 1 e extracao/divisao temporal dia 2"
# Descomente para enviar ao repositório se o remote estiver autenticado:
# !git push origin main